<a href="https://colab.research.google.com/github/eayrault/Projet-Audio/blob/main/projet_audio_eliot_ayrault.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
%%capture
!pip install transformers torch soundfile accelerate datasets

In [21]:
import soundfile as sf
import numpy as np

audio_path = '/content/testaudio.mp3'

try:
    # Read audio file
    audio_data, sampling_rate = sf.read(audio_path, dtype='float32')
    print(f"Audio loaded successfully from {audio_path}")
    print(f"Sampling Rate: {sampling_rate} Hz")
    print(f"Duration: {len(audio_data) / sampling_rate:.2f} seconds")

except FileNotFoundError:
    print(f"Error: Audio file not found at {audio_path}")
    audio_data = None
    sampling_rate = None
except Exception as e:
    print(f"An error occurred while loading the audio file: {e}")
    audio_data = None
    sampling_rate = None


Audio loaded successfully from /content/testaudio.mp3
Sampling Rate: 24000 Hz
Duration: 2.21 seconds


In [22]:
from transformers import pipeline
import numpy as np

if audio_data is not None:
    print("Initializing Speech-to-Text pipeline...")

    # Convert stereo audio to mono by taking the mean across channels if necessary
    # audio_data from soundfile.read is typically (num_samples, num_channels)
    if audio_data.ndim > 1:
        mono_audio_data = np.mean(audio_data, axis=1)
    else:
        mono_audio_data = audio_data

    asr_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0) # Use device=0 for GPU if available

    # Transcribe the audio
    # Pass audio data as a dictionary with 'array' and 'sampling_rate' keys
    transcription = asr_pipeline({"sampling_rate": sampling_rate, "array": mono_audio_data}, return_timestamps=True)
    transcribed_text = transcription['text']
    print("\nTranscription: ")
    print(transcribed_text)
else:
    transcribed_text = ""
    print("Skipping ASR as audio data could not be loaded.")

Initializing Speech-to-Text pipeline...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


Transcription: 
 Follow for part two.


In [23]:
if transcribed_text:
    print("Initializing Text-to-Text pipeline...")
    # Using a small text generation model for demonstration
    text_generator = pipeline("text-generation", model="distilgpt2", device=0) # Use device=0 for GPU if available

    # Generate a response based on the transcribed text
    # Limiting max_new_tokens to avoid very long responses and adding instruction for generating a short response
    prompt = f"Based on the following text, provide a short, response: {transcribed_text}\nResponse:"
    generated_response = text_generator(prompt, max_new_tokens=50, num_return_sequences=1, truncation=True)
    llm_response_text = generated_response[0]['generated_text'].replace(prompt, '').strip()
    print("\nGenerated Response: ")
    print(llm_response_text)
else:
    llm_response_text = ""
    print("Skipping Text-to-Text as no transcribed text is available.")


Initializing Text-to-Text pipeline...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Response: 
I agree with this message and will work on it.
The project is still in its early stages. There is still time to build it.
In the meantime, please consider reading the comments.


In [25]:
if llm_response_text:
    print("Initializing Text-to-Speech pipeline...")
    tts_pipeline = pipeline("text-to-speech", model="facebook/mms-tts-eng", device=0) # Use device=0 for GPU if available

    # Generate speech from the LLM's response
    speech = tts_pipeline(llm_response_text)

    generated_audio_data = speech['audio']
    generated_sampling_rate = speech['sampling_rate']

    output_audio_path = 'generated_response.wav'
    sf.write(output_audio_path, generated_audio_data, generated_sampling_rate)
    print(f"\nGenerated speech saved to {output_audio_path}")
else:
    output_audio_path = None
    print("Skipping Text-to-Speech as no LLM response is available.")

Initializing Text-to-Speech pipeline...


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]


Generated speech saved to generated_response.wav


In [27]:
from IPython.display import Audio
display(Audio(output_audio_path))

## README du Projet Speech-to-Text-to-Text-to-Speech

Ce projet implémente une chaîne de traitement audio complète : de la transcription vocale (Speech-to-Text) à la génération de texte (Text-to-Text) et enfin à la synthèse vocale (Text-to-Speech).

### 1. Prérequis

Assurez-vous d'avoir les dépendances nécessaires installées. La première cellule du notebook (`!pip install ...`) s'en charge automatiquement.

### 2. Fichier Audio d'Entrée

Le projet utilise par défaut le fichier audio `testaudio.mp3` situé dans le répertoire `/content/`. Pour changer le fichier audio d'entrée, modifiez la variable `audio_path`:

```python
audio_path = '/content/testaudio.mp3' # Changez 'testaudio.mp3' par le nom de votre fichier
```

### 3. Fonctionnement des Étapes

*   **Récupération de l'Audio**: Charge le fichier `testaudio.mp3` et affiche ses propriétés (fréquence d'échantillonnage, durée).
*   **Speech-to-Text**: Utilise un modèle Whisper pour transcrire l'audio en texte. Le modèle `openai/whisper-small` est utilisé par défaut. Si votre audio est long (plus de 30 secondes), `return_timestamps=True` est activé pour gérer la transcription longue.
*   **Text-to-Text**: Prend le texte transcrit et utilise un modèle de génération de texte (`distilgpt2`) pour créer une réponse. Vous pouvez adapter le `prompt` pour guider la génération de texte.
*   **Text-to-Speech**: Convertit la réponse générée par le modèle de texte en un nouveau fichier audio (`generated_response.wav`) en utilisant un modèle de synthèse vocale (`facebook/mms-tts-eng`).
*   **Lecture de l'Audio**: Joue le fichier audio généré directement dans le notebook.

### 4. Personnalisation

*   **Modèles**: Vous pouvez expérimenter avec d'autres modèles Hugging Face pour chaque étape (Speech-to-Text, Text-to-Text, Text-to-Speech) en modifiant les noms des modèles dans les pipelines correspondants.
*   **Prompt LLM**: Ajustez le `prompt` dans l'étape Text-to-Text pour obtenir des types de réponses différents du LLM.

### 5. Exécution

Exécutez simplement les cellules du notebook séquentiellement pour voir le pipeline en action.